<table align="left">
  <td>
    <a href="https://colab.research.google.com/" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab" title="Abrir y ejecutar en Google Colaboratory"/></a>
  </td>
  <td>
    <a target="_blank" href="https://kaggle.com/kernels/welcome"><img src="https://kaggle.com/static/images/open-in-kaggle.svg" alt="Abrir en Kaggle" title="Abrir y ejecutar en Kaggle"/></a>
  </td>
</table>

<br><br>

# Representación, clustering aplicado y confiabilidad del modelo

**Curso:** Aprendizaje Automático — SI7009 - 1 (5553)  
**Sesión 4:** Estructura, clustering y confiabilidad  
**Universidad:** EAFIT  
**Profesor:** Marco Teran

---

## Propósito

Este notebook cierra el curso con una idea metodológica central:

> Un buen modelo no solo predice. También debe poder leerse, segmentarse, auditarse, calibrarse y monitorearse.

El flujo completo trabaja tres actos:

1. **Datos sintéticos controlados:** geometría, scaling, PCA, K-Means, outliers globales y rarezas locales.
2. **Online Retail:** RFM, PCA, t-SNE, UMAP opcional, K-Means, MiniBatchKMeans, silhouette, perfiles de segmentos y outliers.
3. **Credit Card Fraud:** train/calibration/test, Brier score, calibration curves, threshold policy y drift básico.

## Contrato metodológico

- No gráfico sin interpretación.
- No cluster sin perfil.
- No embedding como prueba definitiva de estructura.
- No silhouette como única regla para elegir `k`.
- No outlier como error automático.
- No score como probabilidad sin calibration.
- No threshold tuning sobre test.
- No drift como explicación completa.

## Instalación sugerida

En Colab o local, si falta alguna dependencia:

```bash
pip install numpy pandas matplotlib scikit-learn scipy openpyxl umap-learn
```

`umap-learn` es opcional. Si no está instalado, el notebook continúa con PCA y t-SNE.

In [ ]:
# =========================
# Imports and configuration
# =========================

import os
import glob
import json
import time
import random
import warnings
import importlib
from pathlib import Path
from typing import Any, Dict, List, Optional, Sequence, Tuple, Union

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import make_blobs
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, MiniBatchKMeans
from sklearn.metrics import silhouette_score
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import calibration_curve
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    brier_score_loss,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
)

try:
    from scipy import stats
    SCIPY_AVAILABLE = True
except Exception:
    stats = None
    SCIPY_AVAILABLE = False

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

RANDOM_STATE = 42
MAX_TSNE_SAMPLE = 2500
MAX_UMAP_SAMPLE = 5000
MAX_LOF_SAMPLE = 5000
MAX_SILHOUETTE_SAMPLE = 5000

np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")

plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.linestyle"] = "--"
plt.rcParams["grid.alpha"] = 0.4
plt.rcParams["axes.titlesize"] = 12
plt.rcParams["axes.labelsize"] = 11

PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"

evidence_registry: List[Dict[str, Any]] = []
audit_status: Dict[str, Any] = {
    "dummy_geometry_created": False,
    "dummy_outlier_created": False,
    "online_retail_loaded": False,
    "rfm_table_created": False,
    "scaling_applied_before_distance_methods": False,
    "pca_completed": False,
    "tsne_completed_or_documented_skip": False,
    "umap_completed_or_gracefully_skipped": False,
    "kmeans_completed": False,
    "minibatch_kmeans_completed": False,
    "silhouette_completed": False,
    "cluster_profiles_created": False,
    "outlier_models_completed": False,
    "creditcard_loaded_or_fallback_documented": False,
    "calibration_curve_completed": False,
    "brier_score_completed": False,
    "drift_diagnostic_completed": False,
    "no_test_tuning_declared": True,
    "concept_coverage_table_completed": False,
}

In [ ]:
# =========================
# Helper functions
# =========================

def print_section(title: str) -> None:
    line = "=" * len(title)
    print(f"\n{title}\n{line}")


def optional_import(module_name: str, package_hint: Optional[str] = None):
    try:
        module = importlib.import_module(module_name)
        return True, module
    except Exception as exc:
        print(f"Optional dependency not available: {module_name}")
        print(f"Install hint: pip install {package_hint or module_name}")
        print(f"Reason: {exc}")
        return False, None


def get_package_version(module_name: str) -> str:
    try:
        module = importlib.import_module(module_name)
        return getattr(module, "__version__", "available_version_unknown")
    except Exception:
        return "not_available"


def build_dependency_report() -> pd.DataFrame:
    specs = [
        ("numpy", "numpy", True, "Operaciones numéricas"),
        ("pandas", "pandas", True, "Manipulación de datos"),
        ("matplotlib", "matplotlib", True, "Visualización"),
        ("scikit-learn", "sklearn", True, "Modelado y métricas"),
        ("scipy", "scipy", False, "Pruebas estadísticas opcionales"),
        ("openpyxl", "openpyxl", False, "Lectura de Excel"),
        ("umap-learn", "umap", False, "UMAP opcional"),
    ]
    rows = []
    for package, import_name, required, role in specs:
        version = get_package_version(import_name)
        rows.append({
            "package": package,
            "import_name": import_name,
            "required": required,
            "available": version != "not_available",
            "version": version,
            "role": role,
        })
    return pd.DataFrame(rows)


def register_evidence(concept: str, section: str, dataset: str, artifact_type: str,
                      artifact_name: str, status: str = "completed", notes: str = "") -> None:
    evidence_registry.append({
        "concept": concept,
        "section": section,
        "dataset": dataset,
        "artifact_type": artifact_type,
        "artifact_name": artifact_name,
        "status": status,
        "notes": notes,
    })


def evidence_table() -> pd.DataFrame:
    return pd.DataFrame(evidence_registry)


def update_audit_flag(flag_name: str, value: bool, strict: bool = False) -> None:
    if strict and flag_name not in audit_status:
        raise KeyError(f"Unknown audit flag: {flag_name}")
    audit_status[flag_name] = bool(value)


def audit_table() -> pd.DataFrame:
    return pd.DataFrame([{"check": key, "passed": value} for key, value in audit_status.items()])


def resolve_dataset_path(candidate_names: Sequence[str],
                         extra_search_dirs: Optional[Sequence[Union[str, Path]]] = None,
                         recursive: bool = True) -> Optional[Path]:
    search_dirs = [
        PROJECT_ROOT,
        DATA_DIR,
        Path("/content"),
        Path("/content/data"),
        Path("/kaggle/input"),
        Path("/kaggle/working"),
    ]
    if extra_search_dirs is not None:
        search_dirs.extend([Path(path) for path in extra_search_dirs])

    matches = []
    for directory in search_dirs:
        for candidate in candidate_names:
            patterns = [directory / candidate]
            if recursive:
                patterns.append(directory / "**" / candidate)
            for pattern in patterns:
                matches.extend(glob.glob(str(pattern), recursive=recursive))
    matches = sorted(set(matches))
    return Path(matches[0]) if matches else None


def audit_dataframe(df: pd.DataFrame, name: str = "dataframe") -> pd.DataFrame:
    print_section(f"Audit for {name}")
    print(f"Rows: {df.shape[0]:,}")
    print(f"Columns: {df.shape[1]:,}")
    print(f"Duplicated rows: {df.duplicated().sum():,}")
    audit = pd.DataFrame({
        "column": df.columns,
        "dtype": [str(dtype) for dtype in df.dtypes],
        "missing_count": df.isna().sum().values,
        "missing_pct": df.isna().mean().values * 100,
        "n_unique": [df[col].nunique(dropna=True) for col in df.columns],
    })
    return audit.sort_values(["missing_pct", "column"], ascending=[False, True]).reset_index(drop=True)


def summarize_numeric_columns(df: pd.DataFrame, columns: Optional[Sequence[str]] = None) -> pd.DataFrame:
    if columns is None:
        numeric_df = df.select_dtypes(include=[np.number])
    else:
        numeric_df = df[list(columns)].select_dtypes(include=[np.number])
    if numeric_df.empty:
        return pd.DataFrame()
    summary = numeric_df.describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]).T
    summary["missing_pct"] = numeric_df.isna().mean() * 100
    summary["skew"] = numeric_df.skew(numeric_only=True)
    return summary


def safe_sample(obj: Union[pd.DataFrame, np.ndarray], max_n: int,
                random_state: int = RANDOM_STATE):
    n_rows = len(obj)
    if n_rows <= max_n:
        return obj
    rng = np.random.default_rng(random_state)
    idx = rng.choice(n_rows, size=max_n, replace=False)
    if isinstance(obj, pd.DataFrame):
        return obj.iloc[idx].copy()
    return obj[idx]


def plot_2d_projection(x, y=None, title="Proyección 2D", xlabel="Componente 1",
                       ylabel="Componente 2", alpha=0.75, figsize=(8, 5)):
    arr = np.asarray(x)
    plt.figure(figsize=figsize)
    if y is None:
        plt.scatter(arr[:, 0], arr[:, 1], alpha=alpha, s=22)
    else:
        labels = np.asarray(y)
        for label in np.unique(labels):
            mask = labels == label
            plt.scatter(arr[mask, 0], arr[mask, 1], alpha=alpha, s=22, label=str(label))
        plt.legend(title="Grupo", loc="best")
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.tight_layout()
    plt.show()


def plot_histogram(values, title: str, xlabel: str, bins: int = 40):
    arr = np.asarray(values)
    arr = arr[~pd.isna(arr)]
    plt.figure(figsize=(8, 5))
    plt.hist(arr, bins=bins, alpha=0.85)
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel("Frecuencia")
    plt.tight_layout()
    plt.show()


def plot_bar(values, labels, title: str, xlabel: str, ylabel: str):
    plt.figure(figsize=(8, 5))
    plt.bar(labels, values)
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.tight_layout()
    plt.show()


def evaluate_kmeans_range(x_scaled: np.ndarray, k_values: Sequence[int],
                          random_state: int = RANDOM_STATE,
                          silhouette_sample_size: int = MAX_SILHOUETTE_SAMPLE) -> pd.DataFrame:
    rows = []
    x_for_silhouette = safe_sample(x_scaled, max_n=silhouette_sample_size, random_state=random_state)
    for k in k_values:
        model = KMeans(n_clusters=k, n_init=20, random_state=random_state)
        labels = model.fit_predict(x_scaled)
        if len(x_for_silhouette) < len(x_scaled):
            sampled_labels = model.predict(x_for_silhouette)
            sil = silhouette_score(x_for_silhouette, sampled_labels)
        else:
            sil = silhouette_score(x_scaled, labels)
        rows.append({
            "k": k,
            "inertia": model.inertia_,
            "silhouette": sil,
            "cluster_min_size": pd.Series(labels).value_counts().min(),
            "cluster_max_size": pd.Series(labels).value_counts().max(),
        })
    return pd.DataFrame(rows)


def plot_kmeans_diagnostics(results: pd.DataFrame) -> None:
    fig, ax1 = plt.subplots(figsize=(8, 5))
    ax1.plot(results["k"], results["inertia"], marker="o")
    ax1.set_xlabel("Número de clusters k")
    ax1.set_ylabel("Inertia")
    ax1.set_title("Diagnóstico de K-Means: inertia y silhouette")
    ax2 = ax1.twinx()
    ax2.plot(results["k"], results["silhouette"], marker="s", linestyle="--")
    ax2.set_ylabel("Silhouette")
    fig.tight_layout()
    plt.show()


UMAP_AVAILABLE, umap_module = optional_import("umap", "umap-learn")
dependency_report = build_dependency_report()
dependency_report

# Parte I — Datos sintéticos: geometría, scaling, PCA y K-Means

Primero usamos datos sintéticos para aislar mecanismos sin ruido de negocio. Esto permite ver con claridad cómo la escala altera distancias, cómo PCA proyecta información y cómo K-Means particiona el espacio.

In [ ]:
def make_dummy_geometry_dataset(n_samples: int = 700, random_state: int = RANDOM_STATE) -> pd.DataFrame:
    x, y = make_blobs(
        n_samples=n_samples,
        centers=4,
        cluster_std=[0.8, 1.1, 0.6, 1.3],
        random_state=random_state,
    )
    transform = np.array([[1.7, -0.6], [0.4, 0.9]])
    x_aniso = x @ transform
    rng = np.random.default_rng(random_state)
    df = pd.DataFrame(x_aniso, columns=["feature_1", "feature_2"])
    df["feature_large_scale"] = df["feature_1"] * 80 + rng.normal(0, 5, size=n_samples)
    df["true_group"] = y
    return df


dummy_geometry_df = make_dummy_geometry_dataset()
update_audit_flag("dummy_geometry_created", True, strict=True)
register_evidence(
    "Geometría del dato",
    "Dummy geometry",
    "Synthetic geometry dataset",
    "dataframe",
    "dummy_geometry_df",
    notes="Dataset sintético para estudiar escala, PCA y K-Means.",
)

display(dummy_geometry_df.head())
display(audit_dataframe(dummy_geometry_df, "dummy_geometry_df"))
plot_2d_projection(
    dummy_geometry_df[["feature_1", "feature_2"]],
    y=dummy_geometry_df["true_group"],
    title="Dataset sintético: geometría original",
    xlabel="feature_1",
    ylabel="feature_2",
)

### Lectura técnica

Los grupos sintéticos permiten ver estructura geométrica controlada. En datos reales no tenemos etiquetas verdaderas de cluster; por eso aquí las usamos solo para entender el mecanismo.

In [ ]:
dummy_feature_cols = ["feature_1", "feature_2", "feature_large_scale"]
dummy_x_raw = dummy_geometry_df[dummy_feature_cols].copy()
dummy_scaler = StandardScaler()
dummy_x_scaled = dummy_scaler.fit_transform(dummy_x_raw)
dummy_x_scaled_df = pd.DataFrame(dummy_x_scaled, columns=dummy_feature_cols, index=dummy_geometry_df.index)

update_audit_flag("scaling_applied_before_distance_methods", True, strict=True)
register_evidence(
    "Scaling antes de métodos basados en distancia",
    "Dummy geometry",
    "Synthetic geometry dataset",
    "transformation",
    "StandardScaler on dummy_x_raw",
    notes="Escalado aplicado antes de PCA y K-Means.",
)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].scatter(dummy_geometry_df["feature_1"], dummy_geometry_df["feature_large_scale"],
                c=dummy_geometry_df["true_group"], alpha=0.75, s=20)
axes[0].set_title("Antes de escalar")
axes[0].set_xlabel("feature_1")
axes[0].set_ylabel("feature_large_scale")
axes[1].scatter(dummy_x_scaled_df["feature_1"], dummy_x_scaled_df["feature_large_scale"],
                c=dummy_geometry_df["true_group"], alpha=0.75, s=20)
axes[1].set_title("Después de StandardScaler")
axes[1].set_xlabel("feature_1 escalada")
axes[1].set_ylabel("feature_large_scale escalada")
plt.tight_layout()
plt.show()

### Lectura técnica

Una variable de gran escala domina la geometría aunque no sea conceptualmente más importante. Por eso PCA, K-Means, t-SNE, UMAP y LOF deben usar una representación escalada cuando las variables están en unidades diferentes.

In [ ]:
dummy_pca = PCA(n_components=2, random_state=RANDOM_STATE)
dummy_pca_projection = dummy_pca.fit_transform(dummy_x_scaled)
dummy_pca_df = pd.DataFrame(dummy_pca_projection, columns=["PC1", "PC2"], index=dummy_geometry_df.index)
dummy_explained_variance = pd.Series(dummy_pca.explained_variance_ratio_, index=["PC1", "PC2"])

update_audit_flag("pca_completed", True, strict=True)
register_evidence(
    "PCA como representación lineal",
    "Dummy PCA",
    "Synthetic geometry dataset",
    "model_and_projection",
    "dummy_pca_df",
    notes="PCA aplicado sobre features escaladas.",
)

display(dummy_explained_variance.to_frame("explained_variance_ratio"))
plot_2d_projection(
    dummy_pca_df[["PC1", "PC2"]],
    y=dummy_geometry_df["true_group"],
    title="PCA sobre dataset sintético escalado",
    xlabel="PC1",
    ylabel="PC2",
)
plot_bar(
    dummy_explained_variance.values,
    dummy_explained_variance.index,
    "Varianza explicada por componente principal",
    "Componente",
    "Proporción de varianza explicada",
)

### Lectura técnica

PCA conserva direcciones lineales de variabilidad, pero una proyección 2D siempre pierde información. Cercanía visual no significa cercanía semántica perfecta.

In [ ]:
dummy_k = 4
dummy_kmeans_raw = KMeans(n_clusters=dummy_k, n_init=20, random_state=RANDOM_STATE)
dummy_labels_raw = dummy_kmeans_raw.fit_predict(dummy_x_raw)

dummy_kmeans_scaled = KMeans(n_clusters=dummy_k, n_init=20, random_state=RANDOM_STATE)
dummy_labels_scaled = dummy_kmeans_scaled.fit_predict(dummy_x_scaled)

register_evidence(
    "K-Means y sensibilidad al escalado",
    "Dummy K-Means",
    "Synthetic geometry dataset",
    "model_comparison",
    "dummy_kmeans_raw vs dummy_kmeans_scaled",
    notes="Comparación de K-Means antes y después de StandardScaler.",
)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
axes[0].scatter(dummy_geometry_df["feature_1"], dummy_geometry_df["feature_2"],
                c=dummy_geometry_df["true_group"], alpha=0.75, s=20)
axes[0].set_title("Construcción original")
axes[1].scatter(dummy_geometry_df["feature_1"], dummy_geometry_df["feature_2"],
                c=dummy_labels_raw, alpha=0.75, s=20)
axes[1].set_title("K-Means sin escalar")
axes[2].scatter(dummy_geometry_df["feature_1"], dummy_geometry_df["feature_2"],
                c=dummy_labels_scaled, alpha=0.75, s=20)
axes[2].set_title("K-Means con StandardScaler")
for ax in axes:
    ax.set_xlabel("feature_1")
    ax.set_ylabel("feature_2")
plt.tight_layout()
plt.show()

dummy_kmeans_range_results = evaluate_kmeans_range(dummy_x_scaled, range(2, 9))
register_evidence(
    "Selección crítica de k",
    "Dummy K-Means",
    "Synthetic geometry dataset",
    "diagnostic_table",
    "dummy_kmeans_range_results",
    notes="Evaluación de k con inertia, silhouette y tamaños de clusters.",
)
display(dummy_kmeans_range_results)
plot_kmeans_diagnostics(dummy_kmeans_range_results)

### Lectura técnica

K-Means no descubre una verdad oculta; propone una partición según una geometría. La selección de `k` no debe depender solo de una curva: se combina con tamaño de grupos e interpretación.

# Parte II — Outliers sintéticos: Isolation Forest y LOF

Un outlier no es automáticamente un error ni un fraude. Es una señal para investigar. Aquí contrastamos outliers globales y rarezas locales.

In [ ]:
def make_dummy_outlier_dataset(random_state: int = RANDOM_STATE) -> pd.DataFrame:
    rng = np.random.default_rng(random_state)
    dense_cluster = rng.normal(loc=[0, 0], scale=[0.45, 0.45], size=(350, 2))
    sparse_cluster = rng.normal(loc=[4, 4], scale=[1.0, 1.0], size=(180, 2))
    global_outliers = np.array([[8.5, 8.0], [-5.5, 5.0], [6.5, -4.5], [-4.5, -4.0]])
    local_outliers = np.array([[4.2, 1.2], [1.4, 4.4], [5.8, 2.1]])
    x = np.vstack([dense_cluster, sparse_cluster, global_outliers, local_outliers])
    labels = (
        ["dense_cluster"] * len(dense_cluster)
        + ["sparse_cluster"] * len(sparse_cluster)
        + ["global_outlier"] * len(global_outliers)
        + ["local_outlier"] * len(local_outliers)
    )
    df = pd.DataFrame(x, columns=["x1", "x2"])
    df["construction_label"] = labels
    return df


def fit_isolation_forest(x: np.ndarray, contamination: Union[str, float] = "auto",
                         random_state: int = RANDOM_STATE):
    model = IsolationForest(n_estimators=200, contamination=contamination, random_state=random_state)
    labels = model.fit_predict(x)
    scores = model.decision_function(x)
    return model, labels, scores


def fit_lof_on_sample(x: np.ndarray, max_n: int = MAX_LOF_SAMPLE,
                      n_neighbors: int = 25, contamination: Union[str, float] = "auto",
                      random_state: int = RANDOM_STATE) -> Dict[str, Any]:
    x_sample = safe_sample(x, max_n=max_n, random_state=random_state)
    model = LocalOutlierFactor(n_neighbors=n_neighbors, contamination=contamination)
    labels = model.fit_predict(x_sample)
    scores = model.negative_outlier_factor_
    return {"model": model, "x_sample": x_sample, "labels": labels, "scores": scores}


dummy_outlier_df = make_dummy_outlier_dataset()
update_audit_flag("dummy_outlier_created", True, strict=True)
register_evidence(
    "Outlier global vs rareza local",
    "Dummy outliers",
    "Synthetic outlier dataset",
    "dataframe",
    "dummy_outlier_df",
    notes="Dataset sintético para diferenciar rarezas globales y locales.",
)

display(dummy_outlier_df["construction_label"].value_counts().to_frame("count"))
plot_2d_projection(
    dummy_outlier_df[["x1", "x2"]],
    y=dummy_outlier_df["construction_label"],
    title="Dataset sintético de outliers",
    xlabel="x1",
    ylabel="x2",
)

In [ ]:
dummy_outlier_x = dummy_outlier_df[["x1", "x2"]].to_numpy()

dummy_iforest_model, dummy_iforest_labels, dummy_iforest_scores = fit_isolation_forest(
    dummy_outlier_x, contamination=0.025, random_state=RANDOM_STATE
)
dummy_outlier_iforest_df = dummy_outlier_df.copy()
dummy_outlier_iforest_df["iforest_score"] = dummy_iforest_scores
dummy_outlier_iforest_df["iforest_is_outlier"] = dummy_iforest_labels == -1

register_evidence(
    "Isolation Forest",
    "Dummy outliers",
    "Synthetic outlier dataset",
    "model_and_scores",
    "dummy_iforest_model, dummy_outlier_iforest_df",
    notes="Isolation Forest aplicado para detectar rarezas globales.",
)

plot_2d_projection(
    dummy_outlier_iforest_df[["x1", "x2"]],
    y=dummy_outlier_iforest_df["iforest_is_outlier"].astype(str),
    title="Isolation Forest: outliers sintéticos",
    xlabel="x1",
    ylabel="x2",
)
display(dummy_outlier_iforest_df.sort_values("iforest_score").head(10))

In [ ]:
dummy_lof_result = fit_lof_on_sample(
    dummy_outlier_x, max_n=len(dummy_outlier_x), n_neighbors=25,
    contamination=0.025, random_state=RANDOM_STATE
)
dummy_outlier_lof_df = dummy_outlier_df.copy()
dummy_outlier_lof_df["lof_score"] = dummy_lof_result["scores"]
dummy_outlier_lof_df["lof_is_outlier"] = dummy_lof_result["labels"] == -1

register_evidence(
    "LOF como rareza local",
    "Dummy outliers",
    "Synthetic outlier dataset",
    "model_and_scores",
    "dummy_lof_result, dummy_outlier_lof_df",
    notes="LOF aplicado para detectar rarezas locales.",
)

plot_2d_projection(
    dummy_outlier_lof_df[["x1", "x2"]],
    y=dummy_outlier_lof_df["lof_is_outlier"].astype(str),
    title="LOF: outliers locales sintéticos",
    xlabel="x1",
    ylabel="x2",
)
display(dummy_outlier_lof_df.sort_values("lof_score").head(10))

In [ ]:
dummy_outlier_comparison_df = dummy_outlier_df.copy()
dummy_outlier_comparison_df["iforest_is_outlier"] = dummy_outlier_iforest_df["iforest_is_outlier"]
dummy_outlier_comparison_df["lof_is_outlier"] = dummy_outlier_lof_df["lof_is_outlier"]
dummy_outlier_comparison_df["outlier_agreement"] = np.select(
    [
        dummy_outlier_comparison_df["iforest_is_outlier"] & dummy_outlier_comparison_df["lof_is_outlier"],
        dummy_outlier_comparison_df["iforest_is_outlier"] & ~dummy_outlier_comparison_df["lof_is_outlier"],
        ~dummy_outlier_comparison_df["iforest_is_outlier"] & dummy_outlier_comparison_df["lof_is_outlier"],
    ],
    ["both_methods", "iforest_only", "lof_only"],
    default="neither",
)

register_evidence(
    "Outliers como priorización de investigación",
    "Dummy outliers",
    "Synthetic outlier dataset",
    "comparison_table",
    "dummy_outlier_comparison_df",
    notes="Comparación de señales de Isolation Forest y LOF.",
)

plot_2d_projection(
    dummy_outlier_comparison_df[["x1", "x2"]],
    y=dummy_outlier_comparison_df["outlier_agreement"],
    title="Comparación Isolation Forest vs LOF",
    xlabel="x1",
    ylabel="x2",
)
display(dummy_outlier_comparison_df.groupby(["construction_label", "outlier_agreement"]).size().reset_index(name="count"))
update_audit_flag("outlier_models_completed", True, strict=True)

### Lectura técnica

Isolation Forest y LOF no responden exactamente la misma pregunta. El primero tiende a detectar rarezas globales; el segundo detecta rarezas locales. En ambos casos, la salida es una prioridad de investigación, no una sentencia.

# Parte III — Online Retail: RFM, representación, clustering y outliers

Esta sección se ejecuta si el archivo `Online Retail.xlsx` o `Online Retail.csv` está disponible en `data/`, `/content`, `/kaggle/input` u otra ruta común.

In [ ]:
ONLINE_RETAIL_CANDIDATE_NAMES = [
    "Online Retail.xlsx", "Online Retail.csv", "online_retail.xlsx", "online_retail.csv",
    "OnlineRetail.xlsx", "OnlineRetail.csv", "online-retail.xlsx", "online-retail.csv",
]

online_retail_path = resolve_dataset_path(ONLINE_RETAIL_CANDIDATE_NAMES, recursive=True)
ONLINE_RETAIL_AVAILABLE = online_retail_path is not None

print_section("Online Retail dataset discovery")
print(f"Dataset encontrado en: {online_retail_path}" if ONLINE_RETAIL_AVAILABLE else "No se encontró Online Retail.")


def load_online_retail_file(path: Union[str, Path]) -> pd.DataFrame:
    path = Path(path)
    if path.suffix.lower() in [".xlsx", ".xls"]:
        return pd.read_excel(path)
    if path.suffix.lower() == ".csv":
        for encoding in ["utf-8", "latin1", "ISO-8859-1"]:
            try:
                return pd.read_csv(path, encoding=encoding)
            except Exception:
                pass
    raise ValueError(f"No se pudo leer el archivo: {path}")


def standardize_online_retail_columns(df: pd.DataFrame) -> pd.DataFrame:
    canonical_map = {
        "invoiceno": "InvoiceNo",
        "invoice": "InvoiceNo",
        "stockcode": "StockCode",
        "description": "Description",
        "quantity": "Quantity",
        "invoicedate": "InvoiceDate",
        "date": "InvoiceDate",
        "unitprice": "UnitPrice",
        "price": "UnitPrice",
        "customerid": "CustomerID",
        "customer": "CustomerID",
        "country": "Country",
    }
    renamed = {}
    for col in df.columns:
        normalized = str(col).strip().replace(" ", "").replace("_", "").lower()
        if normalized in canonical_map:
            renamed[col] = canonical_map[normalized]
    out = df.rename(columns=renamed).copy()
    out.columns = [str(c).strip() for c in out.columns]
    return out


def validate_online_retail_schema(df: pd.DataFrame) -> Tuple[bool, List[str]]:
    required = ["InvoiceNo", "Quantity", "InvoiceDate", "UnitPrice", "CustomerID", "Country"]
    missing = [c for c in required if c not in df.columns]
    return len(missing) == 0, missing


if ONLINE_RETAIL_AVAILABLE:
    online_retail_raw = standardize_online_retail_columns(load_online_retail_file(online_retail_path))
    schema_ok, missing_cols = validate_online_retail_schema(online_retail_raw)
    if not schema_ok:
        ONLINE_RETAIL_AVAILABLE = False
        print(f"Columnas faltantes: {missing_cols}")
    else:
        update_audit_flag("online_retail_loaded", True, strict=True)
        register_evidence(
            "Carga de Online Retail",
            "Online Retail loading",
            "Online Retail",
            "dataframe",
            "online_retail_raw",
            notes="Dataset cargado y columnas estandarizadas.",
        )
        display(online_retail_raw.head())
        display(audit_dataframe(online_retail_raw, "online_retail_raw"))
else:
    online_retail_raw = None
    register_evidence(
        "Carga de Online Retail",
        "Online Retail loading",
        "Online Retail",
        "dataset_status",
        "online_retail_path",
        status="missing",
        notes="No se encontró el dataset real.",
    )

In [ ]:
def build_rfm_table(retail_df: pd.DataFrame) -> pd.DataFrame:
    df = retail_df.copy()
    df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"], errors="coerce")
    df["InvoiceNo"] = df["InvoiceNo"].astype(str)
    df = df.dropna(subset=["CustomerID", "InvoiceDate", "Quantity", "UnitPrice"]).copy()
    df = df[
        (df["Quantity"] > 0)
        & (df["UnitPrice"] > 0)
        & (~df["InvoiceNo"].str.startswith("C", na=False))
    ].copy()
    df["line_total"] = df["Quantity"] * df["UnitPrice"]
    snapshot_date = df["InvoiceDate"].max() + pd.Timedelta(days=1)
    rfm = (
        df.groupby("CustomerID")
        .agg(
            recency_days=("InvoiceDate", lambda values: (snapshot_date - values.max()).days),
            frequency=("InvoiceNo", "nunique"),
            monetary_value=("line_total", "sum"),
            total_items=("Quantity", "sum"),
            avg_basket_value=("line_total", "mean"),
        )
        .reset_index()
        .rename(columns={"CustomerID": "customer_id"})
    )
    return rfm


def prepare_rfm_model_features(rfm_df: pd.DataFrame):
    raw = rfm_df[["recency_days", "frequency", "monetary_value"]].copy()
    model = pd.DataFrame(index=rfm_df.index)
    model["log_recency_days"] = np.log1p(raw["recency_days"])
    model["log_frequency"] = np.log1p(raw["frequency"])
    model["log_monetary_value"] = np.log1p(raw["monetary_value"])
    return raw, model


if ONLINE_RETAIL_AVAILABLE:
    online_retail_clean = online_retail_raw.copy()
    online_retail_clean["InvoiceDate"] = pd.to_datetime(online_retail_clean["InvoiceDate"], errors="coerce")
    online_retail_clean["InvoiceNo"] = online_retail_clean["InvoiceNo"].astype(str)
    initial_rows = len(online_retail_clean)
    online_retail_clean = online_retail_clean.dropna(subset=["CustomerID", "InvoiceDate", "Quantity", "UnitPrice"]).copy()
    online_retail_clean = online_retail_clean[
        (online_retail_clean["Quantity"] > 0)
        & (online_retail_clean["UnitPrice"] > 0)
        & (~online_retail_clean["InvoiceNo"].str.startswith("C", na=False))
    ].copy()
    online_retail_clean["line_total"] = online_retail_clean["Quantity"] * online_retail_clean["UnitPrice"]

    cleaning_summary = pd.DataFrame([
        {"item": "initial_rows", "value": initial_rows},
        {"item": "retained_rows", "value": len(online_retail_clean)},
        {"item": "removed_rows", "value": initial_rows - len(online_retail_clean)},
        {"item": "unique_customers", "value": online_retail_clean["CustomerID"].nunique()},
    ])
    display(cleaning_summary)

    rfm_df = build_rfm_table(online_retail_clean)
    update_audit_flag("rfm_table_created", True, strict=True)
    register_evidence(
        "RFM segmentation features",
        "RFM construction",
        "Online Retail",
        "feature_table",
        "rfm_df",
        notes="Tabla cliente-nivel con Recency, Frequency y Monetary.",
    )
    display(rfm_df.head())
    display(audit_dataframe(rfm_df, "rfm_df"))
    display(summarize_numeric_columns(rfm_df, ["recency_days", "frequency", "monetary_value", "total_items", "avg_basket_value"]))

### Lectura técnica

El paso decisivo no es entrenar K-Means; es cambiar la unidad de análisis. Pasamos de transacciones crudas a clientes descritos por comportamiento. Esa representación determina la calidad de la segmentación posterior.

In [ ]:
if ONLINE_RETAIL_AVAILABLE:
    for col, title, xlabel in [
        ("recency_days", "Distribución de Recency", "Días desde la última compra"),
        ("frequency", "Distribución de Frequency", "Número de facturas"),
        ("monetary_value", "Distribución de Monetary", "Valor monetario acumulado"),
    ]:
        plot_histogram(rfm_df[col], title=title, xlabel=xlabel, bins=40)

    rfm_raw_features, rfm_model_features = prepare_rfm_model_features(rfm_df)
    rfm_scaler = StandardScaler()
    rfm_scaled = rfm_scaler.fit_transform(rfm_model_features)
    rfm_scaled_df = pd.DataFrame(rfm_scaled, columns=rfm_model_features.columns, index=rfm_df.index)

    update_audit_flag("scaling_applied_before_distance_methods", True, strict=True)
    register_evidence(
        "Preparación de features para clustering",
        "RFM feature preparation",
        "Online Retail",
        "transformation",
        "rfm_scaled_df",
        notes="RFM transformado con log1p y StandardScaler.",
    )
    display(rfm_scaled_df.describe().T)

### Lectura técnica

RFM tiene colas largas. `log1p` reduce la dominancia de valores extremos y `StandardScaler` estabiliza la geometría para PCA, K-Means y LOF. La interpretación, sin embargo, vuelve siempre a unidades originales.

In [ ]:
if ONLINE_RETAIL_AVAILABLE:
    rfm_pca = PCA(n_components=2, random_state=RANDOM_STATE)
    rfm_pca_projection = rfm_pca.fit_transform(rfm_scaled)
    rfm_pca_df = pd.DataFrame(rfm_pca_projection, columns=["PC1", "PC2"], index=rfm_df.index)
    rfm_pca_explained = pd.Series(rfm_pca.explained_variance_ratio_, index=["PC1", "PC2"])

    update_audit_flag("pca_completed", True, strict=True)
    register_evidence(
        "PCA sobre RFM",
        "RFM representation",
        "Online Retail",
        "embedding",
        "rfm_pca_df",
        notes="PCA aplicado sobre matriz RFM transformada y escalada.",
    )

    display(rfm_pca_explained.to_frame("explained_variance_ratio"))
    plot_2d_projection(rfm_pca_df[["PC1", "PC2"]], title="PCA sobre clientes RFM", xlabel="PC1", ylabel="PC2", alpha=0.55)

    rfm_embedding_sample_df = safe_sample(rfm_scaled_df, MAX_TSNE_SAMPLE, RANDOM_STATE)
    rfm_embedding_sample_index = rfm_embedding_sample_df.index
    tsne_perplexity = min(30, max(5, (len(rfm_embedding_sample_df) - 1) // 3))
    rfm_tsne = TSNE(
        n_components=2,
        perplexity=tsne_perplexity,
        learning_rate="auto",
        init="pca",
        random_state=RANDOM_STATE,
    )
    rfm_tsne_df = pd.DataFrame(
        rfm_tsne.fit_transform(rfm_embedding_sample_df.to_numpy()),
        columns=["TSNE1", "TSNE2"],
        index=rfm_embedding_sample_index,
    )
    update_audit_flag("tsne_completed_or_documented_skip", True, strict=True)
    register_evidence(
        "t-SNE como visualización local",
        "RFM representation",
        "Online Retail",
        "embedding",
        "rfm_tsne_df",
        notes=f"t-SNE aplicado con perplexity={tsne_perplexity}.",
    )
    plot_2d_projection(rfm_tsne_df[["TSNE1", "TSNE2"]], title="t-SNE sobre muestra RFM", xlabel="t-SNE 1", ylabel="t-SNE 2", alpha=0.65)

In [ ]:
if ONLINE_RETAIL_AVAILABLE:
    if UMAP_AVAILABLE:
        rfm_umap_input_df = safe_sample(rfm_scaled_df, MAX_UMAP_SAMPLE, RANDOM_STATE)
        rfm_umap_model = umap_module.UMAP(n_components=2, n_neighbors=30, min_dist=0.1, random_state=RANDOM_STATE)
        rfm_umap_df = pd.DataFrame(
            rfm_umap_model.fit_transform(rfm_umap_input_df.to_numpy()),
            columns=["UMAP1", "UMAP2"],
            index=rfm_umap_input_df.index,
        )
        register_evidence(
            "UMAP como visualización local opcional",
            "RFM representation",
            "Online Retail",
            "embedding",
            "rfm_umap_df",
            notes="UMAP aplicado sobre muestra RFM.",
        )
        plot_2d_projection(rfm_umap_df[["UMAP1", "UMAP2"]], title="UMAP sobre muestra RFM", xlabel="UMAP 1", ylabel="UMAP 2", alpha=0.65)
    else:
        rfm_umap_df = None
        register_evidence(
            "UMAP como visualización local opcional",
            "RFM representation",
            "Online Retail",
            "optional_skip",
            "rfm_umap_df",
            status="skipped_optional",
            notes="UMAP no instalado; el notebook continúa con PCA y t-SNE.",
        )
        print("UMAP no está disponible. Se continúa con PCA y t-SNE.")
    update_audit_flag("umap_completed_or_gracefully_skipped", True, strict=True)

### Lectura técnica

PCA, t-SNE y UMAP son lentes distintos. PCA es lineal y se interpreta por varianza; t-SNE y UMAP ayudan a explorar vecindades locales. Ninguno prueba por sí solo que existan segmentos reales.

In [ ]:
if ONLINE_RETAIL_AVAILABLE:
    rfm_kmeans_range_results = evaluate_kmeans_range(rfm_scaled, range(2, 10), RANDOM_STATE, MAX_SILHOUETTE_SAMPLE)
    rfm_kmeans_range_results["min_cluster_pct"] = rfm_kmeans_range_results["cluster_min_size"] / len(rfm_df) * 100

    update_audit_flag("silhouette_completed", True, strict=True)
    register_evidence(
        "Silhouette como métrica de apoyo",
        "RFM clustering",
        "Online Retail",
        "diagnostic_table",
        "rfm_kmeans_range_results",
        notes="Exploración de k con inertia, silhouette y tamaños mínimos.",
    )

    display(rfm_kmeans_range_results)
    plot_kmeans_diagnostics(rfm_kmeans_range_results)

    feasible_k = rfm_kmeans_range_results[rfm_kmeans_range_results["min_cluster_pct"] >= 2.0].copy()
    if feasible_k.empty:
        feasible_k = rfm_kmeans_range_results.copy()
    selected_k = int(feasible_k.sort_values(["silhouette", "cluster_min_size"], ascending=[False, False]).iloc[0]["k"])

    rfm_kmeans_model = KMeans(n_clusters=selected_k, n_init=30, random_state=RANDOM_STATE)
    rfm_kmeans_labels = rfm_kmeans_model.fit_predict(rfm_scaled)
    rfm_clustered = rfm_df.copy()
    rfm_clustered["cluster"] = rfm_kmeans_labels

    update_audit_flag("kmeans_completed", True, strict=True)
    register_evidence(
        "K-Means como baseline de clustering",
        "RFM clustering",
        "Online Retail",
        "model",
        "rfm_kmeans_model, rfm_clustered",
        notes=f"K-Means entrenado con k={selected_k}.",
    )

    rfm_minibatch_model = MiniBatchKMeans(n_clusters=selected_k, random_state=RANDOM_STATE, batch_size=512, n_init=20)
    rfm_minibatch_labels = rfm_minibatch_model.fit_predict(rfm_scaled)
    rfm_minibatch_comparison = pd.DataFrame([
        {"model": "KMeans", "k": selected_k, "inertia": rfm_kmeans_model.inertia_},
        {"model": "MiniBatchKMeans", "k": selected_k, "inertia": rfm_minibatch_model.inertia_},
    ])

    update_audit_flag("minibatch_kmeans_completed", True, strict=True)
    register_evidence(
        "MiniBatchKMeans para escalabilidad",
        "RFM clustering",
        "Online Retail",
        "model_comparison",
        "rfm_minibatch_comparison",
        notes="Comparación práctica entre K-Means y MiniBatchKMeans.",
    )

    display(rfm_minibatch_comparison)
    display(rfm_clustered.head())

### Lectura técnica

K-Means se usa como baseline, no como verdad definitiva. MiniBatchKMeans es una alternativa práctica para escalar. Silhouette ayuda, pero debe leerse junto con tamaños de clusters y perfiles interpretables.

In [ ]:
def profile_clusters(df_original: pd.DataFrame, labels, profile_columns: Sequence[str], cluster_col: str = "cluster") -> pd.DataFrame:
    profiled = df_original.copy()
    profiled[cluster_col] = labels
    return profiled.groupby(cluster_col)[list(profile_columns)].agg(["count", "mean", "median", "min", "max"]).round(3)


if ONLINE_RETAIL_AVAILABLE:
    rfm_profile_columns = ["recency_days", "frequency", "monetary_value", "total_items", "avg_basket_value"]
    rfm_cluster_profile_multi = profile_clusters(rfm_clustered, rfm_clustered["cluster"], rfm_profile_columns, "cluster_profile_label")
    rfm_cluster_profile_flat = rfm_cluster_profile_multi.copy()
    rfm_cluster_profile_flat.columns = [f"{feature}_{stat}" for feature, stat in rfm_cluster_profile_flat.columns]
    rfm_cluster_profile_flat = rfm_cluster_profile_flat.reset_index().rename(columns={"cluster_profile_label": "cluster"})

    update_audit_flag("cluster_profiles_created", True, strict=True)
    register_evidence(
        "Segmentación accionable mediante perfiles",
        "RFM clustering",
        "Online Retail",
        "profile_table",
        "rfm_cluster_profile_flat",
        notes="Perfil de clusters construido en unidades originales.",
    )

    display(rfm_cluster_profile_flat)

    rfm_pca_cluster_view = rfm_pca_df.copy()
    rfm_pca_cluster_view["cluster"] = rfm_clustered["cluster"].values
    plot_2d_projection(rfm_pca_cluster_view[["PC1", "PC2"]], y=rfm_pca_cluster_view["cluster"], title="Clusters K-Means sobre PCA", xlabel="PC1", ylabel="PC2", alpha=0.65)

    rfm_tsne_cluster_view = rfm_tsne_df.copy()
    rfm_tsne_cluster_view["cluster"] = rfm_clustered.loc[rfm_tsne_df.index, "cluster"].values
    plot_2d_projection(rfm_tsne_cluster_view[["TSNE1", "TSNE2"]], y=rfm_tsne_cluster_view["cluster"], title="Clusters K-Means sobre t-SNE", xlabel="t-SNE 1", ylabel="t-SNE 2", alpha=0.65)

### Lectura técnica

Un cluster empieza a volverse segmento cuando puede explicarse en recency, frequency y monetary en unidades originales. La visualización ayuda a comunicar, pero el perfil es lo que vuelve interpretable el resultado.

In [ ]:
if ONLINE_RETAIL_AVAILABLE:
    rfm_outlier_x = rfm_scaled.copy()
    rfm_outlier_base_df = rfm_clustered.copy()

    rfm_iforest_model, rfm_iforest_labels, rfm_iforest_scores = fit_isolation_forest(
        rfm_outlier_x, contamination=0.02, random_state=RANDOM_STATE
    )
    rfm_outlier_results = rfm_outlier_base_df.copy()
    rfm_outlier_results["iforest_score"] = rfm_iforest_scores
    rfm_outlier_results["iforest_is_outlier"] = rfm_iforest_labels == -1

    register_evidence(
        "Isolation Forest en clientes RFM",
        "RFM outliers",
        "Online Retail",
        "model_and_scores",
        "rfm_iforest_model, rfm_outlier_results",
        notes="Isolation Forest aplicado sobre RFM.",
    )

    rfm_lof_result = fit_lof_on_sample(rfm_outlier_x, MAX_LOF_SAMPLE, 25, 0.02, RANDOM_STATE)
    if len(rfm_outlier_x) <= MAX_LOF_SAMPLE:
        rfm_lof_sample_index = rfm_outlier_base_df.index.to_numpy()
    else:
        rng = np.random.default_rng(RANDOM_STATE)
        sampled_positions = rng.choice(len(rfm_outlier_x), size=MAX_LOF_SAMPLE, replace=False)
        rfm_lof_sample_index = rfm_outlier_base_df.iloc[sampled_positions].index.to_numpy()

    rfm_lof_results = rfm_outlier_base_df.loc[rfm_lof_sample_index].copy()
    rfm_lof_results["rfm_index"] = rfm_lof_results.index
    rfm_lof_results["lof_score"] = rfm_lof_result["scores"]
    rfm_lof_results["lof_is_outlier"] = rfm_lof_result["labels"] == -1

    register_evidence(
        "LOF en clientes RFM",
        "RFM outliers",
        "Online Retail",
        "model_and_scores",
        "rfm_lof_result, rfm_lof_results",
        notes="LOF aplicado sobre muestra RFM.",
    )

    iforest_subset = rfm_outlier_results[["customer_id", "iforest_score", "iforest_is_outlier"]].copy()
    rfm_outlier_comparison = rfm_lof_results.merge(iforest_subset, on="customer_id", how="left", validate="one_to_one")
    rfm_outlier_comparison["rfm_index"] = rfm_outlier_comparison["rfm_index"].astype(int)
    rfm_outlier_comparison["outlier_signal"] = np.select(
        [
            rfm_outlier_comparison["iforest_is_outlier"] & rfm_outlier_comparison["lof_is_outlier"],
            rfm_outlier_comparison["iforest_is_outlier"] & ~rfm_outlier_comparison["lof_is_outlier"],
            ~rfm_outlier_comparison["iforest_is_outlier"] & rfm_outlier_comparison["lof_is_outlier"],
        ],
        ["both_methods", "iforest_only", "lof_only"],
        default="neither",
    )

    rfm_outlier_review_table = rfm_outlier_comparison.copy()
    rfm_outlier_review_table["iforest_rank"] = rfm_outlier_review_table["iforest_score"].rank(method="average", ascending=True)
    rfm_outlier_review_table["lof_rank"] = rfm_outlier_review_table["lof_score"].rank(method="average", ascending=True)
    rfm_outlier_review_table["agreement_bonus"] = np.where(rfm_outlier_review_table["outlier_signal"] == "both_methods", -0.5, 0.0)
    rfm_outlier_review_table["combined_review_rank"] = (
        rfm_outlier_review_table["iforest_rank"] + rfm_outlier_review_table["lof_rank"] + rfm_outlier_review_table["agreement_bonus"]
    ) / 2
    rfm_outlier_review_table = rfm_outlier_review_table.sort_values("combined_review_rank").reset_index(drop=True)

    update_audit_flag("outlier_models_completed", True, strict=True)
    register_evidence(
        "Outliers como priorización de investigación en clientes",
        "RFM outliers",
        "Online Retail",
        "ranking_table",
        "rfm_outlier_review_table",
        notes="Ranking de clientes candidatos a revisión.",
    )

    display(rfm_outlier_review_table[[
        "customer_id", "cluster", "outlier_signal", "recency_days", "frequency",
        "monetary_value", "iforest_score", "lof_score", "combined_review_rank"
    ]].head(20))

    rfm_outlier_pca_signal_view = rfm_pca_df.loc[rfm_outlier_comparison["rfm_index"].to_numpy()].copy()
    rfm_outlier_pca_signal_view["outlier_signal"] = rfm_outlier_comparison["outlier_signal"].to_numpy()
    plot_2d_projection(rfm_outlier_pca_signal_view[["PC1", "PC2"]], y=rfm_outlier_pca_signal_view["outlier_signal"], title="Señales de outlier sobre PCA de clientes RFM", xlabel="PC1", ylabel="PC2", alpha=0.7)

if not ONLINE_RETAIL_AVAILABLE:
    register_evidence(
        "Representación RFM",
        "RFM representation",
        "Online Retail",
        "dataset_status",
        "rfm_scaled_df",
        status="missing_required_dataset",
        notes="No se ejecutó RFM porque Online Retail no está disponible.",
    )
    update_audit_flag("online_retail_loaded", False, strict=True)
    update_audit_flag("rfm_table_created", False, strict=True)

### Lectura técnica

Los outliers en clientes RFM son candidatos a revisión. Pueden representar errores, clientes mayoristas, clientes de alto valor o comportamientos inusuales. No deben eliminarse automáticamente.

# Parte IV — Credit Card Fraud: calibration, threshold y drift

Esta sección se ejecuta con `creditcard.csv` si está disponible. Si no, usa fallbacks sintéticos para preservar la enseñanza conceptual.

In [ ]:
CREDITCARD_CANDIDATE_NAMES = [
    "creditcard.csv", "Credit Card Fraud Detection.csv", "credit_card_fraud.csv",
    "creditcardfraud.csv", "credit-card-fraud.csv",
]

creditcard_path = resolve_dataset_path(CREDITCARD_CANDIDATE_NAMES, recursive=True)
CREDITCARD_AVAILABLE = creditcard_path is not None

print_section("Credit Card Fraud dataset discovery")
print(f"Dataset encontrado en: {creditcard_path}" if CREDITCARD_AVAILABLE else "No se encontró Credit Card Fraud; se usarán fallbacks sintéticos.")


def logit_clip(probabilities, eps: float = 1e-6) -> np.ndarray:
    p = np.asarray(probabilities, dtype=float)
    p = np.clip(p, eps, 1.0 - eps)
    return np.log(p / (1.0 - p))


def fit_platt_calibrator(calibration_scores, y_calibration, random_state: int = RANDOM_STATE) -> LogisticRegression:
    x_calibrator = logit_clip(calibration_scores).reshape(-1, 1)
    calibrator = LogisticRegression(solver="lbfgs", random_state=random_state)
    calibrator.fit(x_calibrator, y_calibration)
    return calibrator


def apply_platt_calibrator(calibrator: LogisticRegression, scores) -> np.ndarray:
    x_scores = logit_clip(scores).reshape(-1, 1)
    return calibrator.predict_proba(x_scores)[:, 1]


def prepare_supervised_splits(df: pd.DataFrame, target_col: str,
                              test_size: float = 0.20, calibration_size: float = 0.20,
                              random_state: int = RANDOM_STATE) -> Dict[str, Any]:
    x = df.drop(columns=[target_col])
    y = df[target_col].astype(int)
    x_train_cal, x_test, y_train_cal, y_test = train_test_split(
        x, y, test_size=test_size, stratify=y, random_state=random_state
    )
    relative_calibration_size = calibration_size / (1.0 - test_size)
    x_train, x_cal, y_train, y_cal = train_test_split(
        x_train_cal, y_train_cal, test_size=relative_calibration_size,
        stratify=y_train_cal, random_state=random_state
    )
    return {"X_train": x_train, "X_cal": x_cal, "X_test": x_test, "y_train": y_train, "y_cal": y_cal, "y_test": y_test}


def evaluate_binary_scores(y_true, y_score, threshold: float = 0.5) -> Dict[str, float]:
    y_true_arr = np.asarray(y_true)
    y_score_arr = np.asarray(y_score)
    y_pred = (y_score_arr >= threshold).astype(int)
    return {
        "roc_auc": roc_auc_score(y_true_arr, y_score_arr),
        "pr_auc": average_precision_score(y_true_arr, y_score_arr),
        "brier_score": brier_score_loss(y_true_arr, y_score_arr),
        "precision": precision_score(y_true_arr, y_pred, zero_division=0),
        "recall": recall_score(y_true_arr, y_pred, zero_division=0),
        "f1": f1_score(y_true_arr, y_pred, zero_division=0),
    }


def compute_calibration_table(y_true, y_score, n_bins: int = 10) -> pd.DataFrame:
    prob_true, prob_pred = calibration_curve(y_true, y_score, n_bins=n_bins, strategy="quantile")
    return pd.DataFrame({
        "mean_predicted_probability": prob_pred,
        "observed_positive_fraction": prob_true,
        "absolute_gap": np.abs(prob_true - prob_pred),
    })


def build_synthetic_calibration_dataset(n_samples: int = 5000, random_state: int = RANDOM_STATE) -> pd.DataFrame:
    rng = np.random.default_rng(random_state)
    true_probability = rng.beta(a=0.8, b=8.0, size=n_samples)
    y = rng.binomial(n=1, p=true_probability)
    overconfident_score = np.clip(true_probability ** 0.55, 1e-5, 1 - 1e-5)
    underconfident_score = np.clip(0.5 * true_probability + 0.05, 1e-5, 1 - 1e-5)
    return pd.DataFrame({
        "true_probability": true_probability,
        "y": y,
        "overconfident_score": overconfident_score,
        "underconfident_score": underconfident_score,
    })


def plot_calibration_curves(curves: Dict[str, pd.DataFrame], title: str):
    plt.figure(figsize=(6, 6))
    plt.plot([0, 1], [0, 1], linestyle="--", label="Calibración perfecta")
    for label, table in curves.items():
        plt.plot(table["mean_predicted_probability"], table["observed_positive_fraction"], marker="o", label=label)
    plt.title(title)
    plt.xlabel("Probabilidad predicha promedio")
    plt.ylabel("Fracción observada de positivos")
    plt.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
if CREDITCARD_AVAILABLE:
    creditcard_raw = pd.read_csv(creditcard_path)
    if "Class" not in creditcard_raw.columns:
        CREDITCARD_AVAILABLE = False
        print("El archivo no tiene columna Class.")
    else:
        update_audit_flag("creditcard_loaded_or_fallback_documented", True, strict=True)
        register_evidence(
            "Carga de Credit Card Fraud",
            "Credit Card reliability",
            "Credit Card Fraud",
            "dataframe",
            "creditcard_raw",
            notes="Dataset supervisado cargado para reliability.",
        )
        display(audit_dataframe(creditcard_raw, "creditcard_raw"))
        display(creditcard_raw["Class"].value_counts(normalize=True).rename("proportion").to_frame())

if CREDITCARD_AVAILABLE:
    creditcard_splits = prepare_supervised_splits(creditcard_raw, "Class")
    X_train, X_cal, X_test = creditcard_splits["X_train"], creditcard_splits["X_cal"], creditcard_splits["X_test"]
    y_train, y_cal, y_test = creditcard_splits["y_train"], creditcard_splits["y_cal"], creditcard_splits["y_test"]

    split_summary = pd.DataFrame([
        {"split": "train", "rows": len(X_train), "positives": int(y_train.sum()), "positive_pct": float(y_train.mean() * 100)},
        {"split": "calibration", "rows": len(X_cal), "positives": int(y_cal.sum()), "positive_pct": float(y_cal.mean() * 100)},
        {"split": "test", "rows": len(X_test), "positives": int(y_test.sum()), "positive_pct": float(y_test.mean() * 100)},
    ])
    register_evidence(
        "Train/calibration/test split sin leakage",
        "Credit Card reliability",
        "Credit Card Fraud",
        "split_table",
        "split_summary",
        notes="Split estratificado para entrenar, calibrar y evaluar.",
    )
    display(split_summary)

In [ ]:
if CREDITCARD_AVAILABLE:
    base_classifier = Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=1000, class_weight="balanced", solver="lbfgs", random_state=RANDOM_STATE)),
    ])
    base_classifier.fit(X_train, y_train)

    base_cal_scores = base_classifier.predict_proba(X_cal)[:, 1]
    base_test_scores = base_classifier.predict_proba(X_test)[:, 1]

    platt_calibrator = fit_platt_calibrator(base_cal_scores, y_cal)
    calibrated_cal_scores = apply_platt_calibrator(platt_calibrator, base_cal_scores)
    calibrated_test_scores = apply_platt_calibrator(platt_calibrator, base_test_scores)

    register_evidence(
        "Calibration sin usar test",
        "Credit Card reliability",
        "Credit Card Fraud",
        "calibrator",
        "platt_calibrator",
        notes="Calibrador ajustado solo con split de calibration.",
    )

    calibration_metrics_table = pd.DataFrame([
        {"split": "calibration", "model": "base_uncalibrated", **evaluate_binary_scores(y_cal, base_cal_scores)},
        {"split": "calibration", "model": "platt_calibrated", **evaluate_binary_scores(y_cal, calibrated_cal_scores)},
        {"split": "test", "model": "base_uncalibrated", **evaluate_binary_scores(y_test, base_test_scores)},
        {"split": "test", "model": "platt_calibrated", **evaluate_binary_scores(y_test, calibrated_test_scores)},
    ])
    update_audit_flag("brier_score_completed", True, strict=True)
    register_evidence(
        "Brier score para calibration",
        "Credit Card reliability",
        "Credit Card Fraud",
        "metric_table",
        "calibration_metrics_table",
        notes="Comparación con Brier score, PR-AUC y ROC-AUC.",
    )
    display(calibration_metrics_table)

    base_test_calibration_table = compute_calibration_table(y_test, base_test_scores)
    calibrated_test_calibration_table = compute_calibration_table(y_test, calibrated_test_scores)

    update_audit_flag("calibration_curve_completed", True, strict=True)
    register_evidence(
        "Calibration curve",
        "Credit Card reliability",
        "Credit Card Fraud",
        "diagnostic_table",
        "base_test_calibration_table, calibrated_test_calibration_table",
        notes="Reliability diagrams en test.",
    )
    plot_calibration_curves(
        {
            "Base no calibrado": base_test_calibration_table,
            "Calibrado tipo Platt": calibrated_test_calibration_table,
        },
        "Calibration curve en test",
    )

### Lectura técnica

PR-AUC y ROC-AUC evalúan ranking. Brier score y calibration curve evalúan si el score puede interpretarse como probabilidad. Un modelo puede ordenar bien y estar mal calibrado.

In [ ]:
if not CREDITCARD_AVAILABLE:
    synthetic_calibration_df = build_synthetic_calibration_dataset()
    synthetic_overconfident_calibration = compute_calibration_table(
        synthetic_calibration_df["y"], synthetic_calibration_df["overconfident_score"]
    )
    synthetic_underconfident_calibration = compute_calibration_table(
        synthetic_calibration_df["y"], synthetic_calibration_df["underconfident_score"]
    )
    synthetic_brier_table = pd.DataFrame([
        {"model": "overconfident_score", "brier_score": brier_score_loss(synthetic_calibration_df["y"], synthetic_calibration_df["overconfident_score"])},
        {"model": "underconfident_score", "brier_score": brier_score_loss(synthetic_calibration_df["y"], synthetic_calibration_df["underconfident_score"])},
    ])

    update_audit_flag("creditcard_loaded_or_fallback_documented", True, strict=True)
    update_audit_flag("calibration_curve_completed", True, strict=True)
    update_audit_flag("brier_score_completed", True, strict=True)

    register_evidence(
        "Brier score para calibration",
        "Credit Card reliability fallback",
        "Synthetic calibration dataset",
        "fallback_metric_table",
        "synthetic_brier_table",
        status="completed_fallback",
        notes="Brier score en fallback sintético.",
    )
    register_evidence(
        "Calibration curve",
        "Credit Card reliability fallback",
        "Synthetic calibration dataset",
        "fallback_diagnostic_table",
        "synthetic_overconfident_calibration, synthetic_underconfident_calibration",
        status="completed_fallback",
        notes="Calibration curves en fallback sintético.",
    )
    display(synthetic_brier_table)
    plot_calibration_curves(
        {
            "Score sobreconfiado": synthetic_overconfident_calibration,
            "Score subconfiado": synthetic_underconfident_calibration,
        },
        "Calibration curve sintética",
    )

In [ ]:
def threshold_sweep_table(y_true, y_score, thresholds: Optional[Sequence[float]] = None) -> pd.DataFrame:
    if thresholds is None:
        thresholds = np.linspace(0.01, 0.99, 99)
    y_true_arr = np.asarray(y_true)
    y_score_arr = np.asarray(y_score)
    rows = []
    for threshold in thresholds:
        y_pred = (y_score_arr >= threshold).astype(int)
        rows.append({
            "threshold": float(threshold),
            "precision": precision_score(y_true_arr, y_pred, zero_division=0),
            "recall": recall_score(y_true_arr, y_pred, zero_division=0),
            "f1": f1_score(y_true_arr, y_pred, zero_division=0),
            "alerts": int(y_pred.sum()),
            "alert_rate_pct": float(y_pred.mean() * 100),
        })
    return pd.DataFrame(rows)


def select_threshold_by_recall_constraint(sweep_df: pd.DataFrame, min_recall: float = 0.80,
                                          objective: str = "precision") -> Dict[str, Any]:
    feasible = sweep_df[sweep_df["recall"] >= min_recall].copy()
    if feasible.empty:
        best = sweep_df.sort_values("recall", ascending=False).iloc[0]
        return {"selected_threshold": float(best["threshold"]), "selection_status": "fallback_highest_recall"}
    best = feasible.sort_values([objective, "threshold"], ascending=[False, False]).iloc[0]
    return {"selected_threshold": float(best["threshold"]), "selection_status": "selected_under_constraint"}


def plot_threshold_tradeoffs(sweep_df: pd.DataFrame, selected_threshold: Optional[float] = None, title: str = "Trade-off de threshold"):
    plt.figure(figsize=(8, 5))
    plt.plot(sweep_df["threshold"], sweep_df["precision"], label="Precision")
    plt.plot(sweep_df["threshold"], sweep_df["recall"], label="Recall")
    plt.plot(sweep_df["threshold"], sweep_df["f1"], label="F1")
    if selected_threshold is not None:
        plt.axvline(selected_threshold, linestyle="--", label=f"Threshold = {selected_threshold:.3f}")
    plt.title(title)
    plt.xlabel("Threshold")
    plt.ylabel("Métrica")
    plt.legend()
    plt.tight_layout()
    plt.show()


if CREDITCARD_AVAILABLE:
    threshold_grid = np.linspace(0.001, 0.999, 150)
    calibrated_cal_threshold_sweep = threshold_sweep_table(y_cal, calibrated_cal_scores, threshold_grid)
    threshold_selection = select_threshold_by_recall_constraint(calibrated_cal_threshold_sweep, min_recall=0.80, objective="precision")
    selected_operating_threshold = threshold_selection["selected_threshold"]
    register_evidence(
        "Threshold tuning sin usar test",
        "Threshold policy",
        "Credit Card Fraud",
        "threshold_policy",
        "selected_operating_threshold",
        notes="Threshold seleccionado solo con split de calibration.",
    )
    display(pd.DataFrame([threshold_selection]))
    plot_threshold_tradeoffs(calibrated_cal_threshold_sweep, selected_operating_threshold, "Calibration split: threshold trade-off")

    threshold_test_comparison = pd.DataFrame([
        {"policy": "default_threshold_0.5", "threshold": 0.5, **evaluate_binary_scores(y_test, calibrated_test_scores, threshold=0.5)},
        {"policy": "selected_on_calibration", "threshold": selected_operating_threshold, **evaluate_binary_scores(y_test, calibrated_test_scores, threshold=selected_operating_threshold)},
    ])
    display(threshold_test_comparison)

else:
    synthetic_threshold_sweep = threshold_sweep_table(
        synthetic_calibration_df["y"], synthetic_calibration_df["overconfident_score"], np.linspace(0.001, 0.999, 150)
    )
    synthetic_threshold_selection = select_threshold_by_recall_constraint(synthetic_threshold_sweep, min_recall=0.80, objective="precision")
    synthetic_selected_threshold = synthetic_threshold_selection["selected_threshold"]
    register_evidence(
        "Threshold tuning sin usar test",
        "Threshold policy fallback",
        "Synthetic calibration dataset",
        "fallback_threshold_policy",
        "synthetic_selected_threshold",
        status="completed_fallback",
        notes="Fallback conceptual: threshold cambia decisiones, no calibration.",
    )
    display(pd.DataFrame([synthetic_threshold_selection]))
    plot_threshold_tradeoffs(synthetic_threshold_sweep, synthetic_selected_threshold, "Fallback sintético: threshold trade-off")

reliability_concept_table = pd.DataFrame([
    {
        "question": "¿El modelo ordena bien los casos riesgosos?",
        "concept": "Ranking",
        "tools": "ROC-AUC, PR-AUC",
        "does_not_answer": "Si 0.80 significa 80% de probabilidad",
    },
    {
        "question": "¿Cuándo convierto un score en alerta?",
        "concept": "Threshold policy",
        "tools": "Threshold sweep, confusion matrix, precision, recall",
        "does_not_answer": "Si el score está calibrado",
    },
    {
        "question": "¿El score puede leerse como probabilidad?",
        "concept": "Calibration",
        "tools": "Calibration curve, Brier score",
        "does_not_answer": "Cuál threshold debe usar el negocio",
    },
])
register_evidence(
    "Threshold tuning no es calibration",
    "Threshold policy",
    "Credit Card Fraud or synthetic fallback",
    "concept_table",
    "reliability_concept_table",
    notes="Tabla conceptual que separa ranking, threshold y calibration.",
)
display(reliability_concept_table)

### Lectura técnica

Mover el threshold cambia la política de decisión: cuántos casos se marcan, cuántos falsos positivos aparecen y cuántos positivos se recuperan. Pero no hace que el score sea una probabilidad mejor calibrada.

In [ ]:
def compute_basic_drift_report(reference_df: pd.DataFrame, current_df: pd.DataFrame, numeric_columns: Sequence[str]) -> pd.DataFrame:
    rows = []
    for col in numeric_columns:
        ref_values = reference_df[col].dropna().to_numpy()
        cur_values = current_df[col].dropna().to_numpy()
        if len(ref_values) == 0 or len(cur_values) == 0:
            continue
        row = {
            "feature": col,
            "reference_mean": float(np.mean(ref_values)),
            "current_mean": float(np.mean(cur_values)),
            "mean_abs_delta": float(abs(np.mean(cur_values) - np.mean(ref_values))),
            "reference_median": float(np.median(ref_values)),
            "current_median": float(np.median(cur_values)),
            "median_abs_delta": float(abs(np.median(cur_values) - np.median(ref_values))),
        }
        if SCIPY_AVAILABLE:
            ks_stat, ks_pvalue = stats.ks_2samp(ref_values, cur_values)
            row["ks_statistic"] = float(ks_stat)
            row["ks_pvalue"] = float(ks_pvalue)
        rows.append(row)
    return pd.DataFrame(rows).sort_values("mean_abs_delta", ascending=False).reset_index(drop=True)


def plot_score_distributions(reference_scores, current_scores, title: str):
    plt.figure(figsize=(8, 5))
    plt.hist(reference_scores, bins=40, alpha=0.55, label="Referencia")
    plt.hist(current_scores, bins=40, alpha=0.55, label="Actual")
    plt.title(title)
    plt.xlabel("Score")
    plt.ylabel("Frecuencia")
    plt.legend()
    plt.tight_layout()
    plt.show()


if CREDITCARD_AVAILABLE:
    drift_numeric_columns = X_train.select_dtypes(include=[np.number]).columns.tolist()
    priority_columns = [c for c in ["Time", "Amount"] if c in drift_numeric_columns]
    v_columns = [c for c in drift_numeric_columns if str(c).startswith("V")][:10]
    drift_columns = priority_columns + v_columns if len(drift_numeric_columns) > 12 else drift_numeric_columns

    feature_drift_report = compute_basic_drift_report(X_train, X_test, drift_columns)
    update_audit_flag("drift_diagnostic_completed", True, strict=True)
    register_evidence(
        "Drift básico en features",
        "Basic drift diagnostics",
        "Credit Card Fraud",
        "drift_table",
        "feature_drift_report",
        notes="Comparación básica entre train y test.",
    )
    display(feature_drift_report.head(15))

    calibrated_train_scores = apply_platt_calibrator(platt_calibrator, base_classifier.predict_proba(X_train)[:, 1])
    score_drift_summary = pd.DataFrame([
        {"sample": "reference_train", "n": len(calibrated_train_scores), "score_mean": float(np.mean(calibrated_train_scores)), "score_median": float(np.median(calibrated_train_scores)), "score_p95": float(np.percentile(calibrated_train_scores, 95))},
        {"sample": "current_test", "n": len(calibrated_test_scores), "score_mean": float(np.mean(calibrated_test_scores)), "score_median": float(np.median(calibrated_test_scores)), "score_p95": float(np.percentile(calibrated_test_scores, 95))},
    ])
    register_evidence(
        "Drift básico en scores",
        "Basic drift diagnostics",
        "Credit Card Fraud",
        "score_drift_table",
        "score_drift_summary",
        notes="Comparación de distribución de scores calibrados entre train y test.",
    )
    display(score_drift_summary)
    plot_score_distributions(calibrated_train_scores, calibrated_test_scores, "Score drift diagnóstico: train vs test")

else:
    rng = np.random.default_rng(RANDOM_STATE)
    synthetic_reference = pd.DataFrame({
        "feature_a": rng.normal(0.0, 1.0, size=3000),
        "feature_b": rng.normal(2.0, 0.8, size=3000),
        "score": rng.beta(0.8, 8.0, size=3000),
    })
    synthetic_current = pd.DataFrame({
        "feature_a": rng.normal(0.4, 1.2, size=3000),
        "feature_b": rng.normal(2.0, 0.8, size=3000),
        "score": rng.beta(1.2, 7.0, size=3000),
    })
    synthetic_drift_report = compute_basic_drift_report(synthetic_reference, synthetic_current, ["feature_a", "feature_b", "score"])
    update_audit_flag("drift_diagnostic_completed", True, strict=True)
    register_evidence(
        "Drift básico en features",
        "Basic drift diagnostics fallback",
        "Synthetic drift dataset",
        "fallback_feature_drift_table",
        "synthetic_drift_report",
        status="completed_fallback",
        notes="Fallback conceptual de drift en features.",
    )
    register_evidence(
        "Drift básico en scores",
        "Basic drift diagnostics fallback",
        "Synthetic drift dataset",
        "fallback_score_drift_plot",
        "synthetic_reference, synthetic_current",
        status="completed_fallback",
        notes="Fallback conceptual de score drift.",
    )
    display(synthetic_drift_report)
    plot_score_distributions(synthetic_reference["score"], synthetic_current["score"], "Score drift sintético")

### Lectura técnica

Drift es una señal de cambio entre un entorno de referencia y uno actual. No prueba por sí solo que el modelo falló, pero sí justifica investigar datos, scores, calibration y desempeño.

# Parte V — Evidencia final, auditoría y cierre

El notebook termina con una auditoría de evidencia, cobertura conceptual, objetos esperados, limitaciones y ejercicios.

In [ ]:
final_evidence_table = evidence_table()
update_audit_flag("concept_coverage_table_completed", True, strict=True)
display(final_evidence_table)

In [ ]:
required_concepts = [
    ("Geometría del dato", True),
    ("Scaling antes de métodos basados en distancia", True),
    ("PCA como representación lineal", True),
    ("PCA sobre RFM", ONLINE_RETAIL_AVAILABLE),
    ("t-SNE como visualización local", ONLINE_RETAIL_AVAILABLE),
    ("UMAP como visualización local opcional", False),
    ("K-Means como baseline de clustering", ONLINE_RETAIL_AVAILABLE),
    ("MiniBatchKMeans para escalabilidad", ONLINE_RETAIL_AVAILABLE),
    ("Silhouette como métrica de apoyo", ONLINE_RETAIL_AVAILABLE),
    ("Segmentación accionable mediante perfiles", ONLINE_RETAIL_AVAILABLE),
    ("Outlier global vs rareza local", True),
    ("Isolation Forest", True),
    ("LOF como rareza local", True),
    ("Isolation Forest en clientes RFM", ONLINE_RETAIL_AVAILABLE),
    ("LOF en clientes RFM", ONLINE_RETAIL_AVAILABLE),
    ("Outliers como priorización de investigación en clientes", ONLINE_RETAIL_AVAILABLE),
    ("Train/calibration/test split sin leakage", CREDITCARD_AVAILABLE),
    ("Calibration sin usar test", CREDITCARD_AVAILABLE),
    ("Brier score para calibration", True),
    ("Calibration curve", True),
    ("Threshold tuning sin usar test", True),
    ("Threshold tuning no es calibration", True),
    ("Drift básico en features", True),
    ("Drift básico en scores", True),
]

coverage_rows = []
for concept, required in required_concepts:
    matches = final_evidence_table[
        final_evidence_table["concept"].str.contains(concept, case=False, regex=False, na=False)
    ] if not final_evidence_table.empty else pd.DataFrame()
    if matches.empty:
        status = "missing"
        artifacts = ""
        datasets = ""
    else:
        statuses = matches["status"].unique().tolist()
        if "completed" in statuses:
            status = "completed"
        elif "completed_fallback" in statuses:
            status = "completed_fallback"
        elif "skipped_optional" in statuses:
            status = "skipped_optional"
        else:
            status = statuses[0]
        artifacts = ", ".join(matches["artifact_name"].astype(str).unique())
        datasets = ", ".join(matches["dataset"].astype(str).unique())
    coverage_rows.append({
        "concept": concept,
        "required": required,
        "coverage_status": status,
        "coverage_passed": (not required) or status in ["completed", "completed_fallback", "skipped_optional"],
        "datasets_found": datasets,
        "artifacts_found": artifacts,
    })

concept_coverage_df = pd.DataFrame(coverage_rows)
required_concept_coverage_passed = bool(concept_coverage_df.loc[concept_coverage_df["required"], "coverage_passed"].all())
display(concept_coverage_df)
display(pd.DataFrame([{
    "required_concept_coverage_passed": required_concept_coverage_passed,
    "required_concepts": int(concept_coverage_df["required"].sum()),
    "required_concepts_passed": int(concept_coverage_df.loc[concept_coverage_df["required"], "coverage_passed"].sum()),
}]))

In [ ]:
def object_exists(object_name: str) -> bool:
    return object_name in globals()

required_object_specs = [
    ("dummy_geometry_df", True, "Dataset sintético de geometría"),
    ("dummy_pca_df", True, "PCA sintético"),
    ("dummy_kmeans_range_results", True, "Diagnóstico K-Means sintético"),
    ("dummy_outlier_df", True, "Dataset sintético de outliers"),
    ("dummy_outlier_comparison_df", True, "Comparación sintética de outliers"),
    ("online_retail_raw", ONLINE_RETAIL_AVAILABLE, "Dataset Online Retail"),
    ("rfm_df", ONLINE_RETAIL_AVAILABLE, "Tabla RFM"),
    ("rfm_scaled_df", ONLINE_RETAIL_AVAILABLE, "RFM escalado"),
    ("rfm_pca_df", ONLINE_RETAIL_AVAILABLE, "PCA sobre RFM"),
    ("rfm_tsne_df", ONLINE_RETAIL_AVAILABLE, "t-SNE sobre RFM"),
    ("rfm_clustered", ONLINE_RETAIL_AVAILABLE, "Clientes con cluster"),
    ("rfm_cluster_profile_flat", ONLINE_RETAIL_AVAILABLE, "Perfil de clusters"),
    ("rfm_outlier_review_table", ONLINE_RETAIL_AVAILABLE, "Ranking outliers RFM"),
    ("creditcard_raw", CREDITCARD_AVAILABLE, "Dataset Credit Card Fraud"),
    ("split_summary", CREDITCARD_AVAILABLE, "Split supervisado"),
    ("base_classifier", CREDITCARD_AVAILABLE, "Modelo base"),
    ("platt_calibrator", CREDITCARD_AVAILABLE, "Calibrador"),
    ("calibration_metrics_table", CREDITCARD_AVAILABLE, "Métricas de calibration"),
]

object_audit_df = pd.DataFrame([
    {
        "object_name": name,
        "required_now": bool(required),
        "exists": object_exists(name),
        "passed": (not required) or object_exists(name),
        "role": role,
    }
    for name, required, role in required_object_specs
])

required_object_audit_passed = bool(object_audit_df.loc[object_audit_df["required_now"], "passed"].all())
display(object_audit_df)

In [ ]:
methodological_audit_df = audit_table()
full_methodological_audit_passed = bool(methodological_audit_df["passed"].all())

# When real datasets are not available, some real-data flags are expected to be false.
# Therefore, we distinguish conceptual completion from real-data completion.
REAL_DATA_COMPLETION_PASSED = bool(ONLINE_RETAIL_AVAILABLE and CREDITCARD_AVAILABLE)
CONCEPTUAL_COMPLETION_PASSED = bool(required_concept_coverage_passed)

NOTEBOOK_FINAL_AUDIT_PASSED = bool(
    required_concept_coverage_passed
    and required_object_audit_passed
)

final_audit_summary = pd.DataFrame([
    {"audit_dimension": "required_concept_coverage", "passed": required_concept_coverage_passed},
    {"audit_dimension": "required_object_audit", "passed": required_object_audit_passed},
    {"audit_dimension": "conceptual_completion", "passed": CONCEPTUAL_COMPLETION_PASSED},
    {"audit_dimension": "real_data_completion", "passed": REAL_DATA_COMPLETION_PASSED},
    {"audit_dimension": "NOTEBOOK_FINAL_AUDIT_PASSED", "passed": NOTEBOOK_FINAL_AUDIT_PASSED},
])
display(methodological_audit_df)
display(final_audit_summary)

## Limitaciones conocidas

1. **Clustering no tiene ground truth:** los clusters no son “correctos” en sentido supervisado.
2. **Silhouette no prueba accionabilidad:** un buen valor puede producir segmentos poco útiles.
3. **PCA, t-SNE y UMAP son lentes, no verdades:** cada método preserva y distorsiona cosas distintas.
4. **RFM simplifica el comportamiento del cliente:** puede ignorar productos, estacionalidad, margen y devoluciones.
5. **Outlier no equivale a fraude o error:** los candidatos requieren investigación.
6. **Calibration depende de la distribución:** un modelo calibrado puede perder calibration si cambia la población.
7. **Drift básico no es MLOps completo:** aquí solo se comparan distribuciones.
8. **Credit Card Fraud se usa para reliability:** el objetivo no es maximizar performance, sino separar ranking, threshold y calibration.

## Ejercicios de extensión

1. Cambia `k` y compara silhouette, tamaños de clusters y perfiles.
2. Repite K-Means sin `log1p` y analiza qué variable domina.
3. Cambia `perplexity` en t-SNE y compara la estabilidad visual.
4. Si tienes UMAP, cambia `n_neighbors` y `min_dist`.
5. Revisa los 10 clientes más raros en `rfm_outlier_review_table`.
6. Cambia la regla de threshold: recall mínimo 90%, maximizar F1 o limitar alertas al 5%.
7. Simula drift más fuerte y observa features y scores.
8. Conecta con tu PI: ¿hay segmentación, outliers, calibration o drift que debas defender?

## Cierre

Esta sesión cierra el curso con una idea profesional:

> Machine Learning no es solo entrenar modelos. Es construir evidencia suficiente para tomar decisiones defendibles.

Esa evidencia puede venir de evaluación supervisada, representación honesta del dato, segmentación interpretable, revisión de anomalías, calibration responsable y diagnóstico básico de drift.